# Challenge completo: pedidos

Tools → Skills → varios Agents → System con decisiones; Environment y Eval sobre la misma declaración.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def quote(units: int, unit_price: int) -> dict:
    if units <= 0 or unit_price < 0:
        raise ValueError("Cantidad o precio inválidos")
    return {"total": units * unit_price}

@toolkit.tool
def approve(total: int) -> dict:
    return {"total": total, "status": "approved"}

@toolkit.tool
def review(total: int) -> dict:
    return {"total": total, "status": "manual_review"}

pricing = toolkit.skill(name="pricing", tools=(quote,))
decisions = toolkit.skill(name="decisions", tools=(approve, review))

def route_order(data):
    return "approve" if data["total"] <= 1000 else "review"

records = ({"units": 2, "unit_price": 100}, {"units": 3, "unit_price": 500})
expected = [{"total": 200, "status": "approved"}, {"total": 1500, "status": "manual_review"}]


def build_orders(backend="native"):
    quoter = toolkit.Agent("quoter", toolkit.Action("quote", "quote"), skills=(pricing,))
    approver = toolkit.Agent("approver", toolkit.Action("approve", "approve"), skills=(decisions,))
    reviewer = toolkit.Agent("reviewer", toolkit.Action("review", "review"), skills=(decisions,))
    return toolkit.System("orders", toolkit.Graph(
        nodes=(("quote", quoter), ("approve", approver), ("review", reviewer)),
        edges=(("quote", route_order), ("approve", "END"), ("review", "END")), start="quote",
    ), backend=backend)

def build_day(backend="native"):
    return toolkit.Environment("day", build_orders(backend),
        records=records, memory={"approved": 0},
        update=lambda memory, record, result: {
            "approved": memory["approved"] + int(result.data["status"] == "approved")
        })

def check_day(result, case):
    valid = result.data["outputs"] == expected and result.data["summary"]["memory"] == {"approved": 1}
    return {"ok": valid, "issues": [] if valid else ["Resultados o memoria incorrectos"],
            "metrics": {"approved": result.data["summary"]["memory"]["approved"]}}

def evaluate_day(unit):
    return toolkit.Eval("orders_acceptance", unit,
        cases=({"input": {}, "expected": {"ok": True}},), checks=(check_day,))



for backend in ("native", "langgraph"):
    day = build_day(backend)
    result = day.run()
    evaluation = evaluate_day(day).run()
    assert result.ok and evaluation.ok and check_day(result, {})["ok"]
    print({"backend": backend, "input": list(records), "process": "quote → decision → approve/review → memory → Eval", "output": result.data})